In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_predict, train_test_split
from sklearn.metrics import roc_auc_score,balanced_accuracy_score, recall_score

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


### Load the embeddings for the C.S.sylv sulcal region

In [2]:
ukb_embeddings = pd.read_csv('/neurospin/dico/data/deep_folding/current/models/Champollion_V0_trained_on_UKB40/SC-sylv_right/11-36-10_85_0/ukb40_random_epoch80_embeddings/full_embeddings.csv', index_col=0)
#ukb_embeddings = pd.read_csv('/neurospin/dico/data/deep_folding/current/models/Champollion_V0/SC-sylv_right/11-43-38_3/ukb40_random_epoch100_embeddings/full_embeddings.csv', index_col=0)
print(ukb_embeddings.shape)
ukb_embeddings.head()

(42433, 256)


,dim1,dim2,dim3,dim4,dim5,dim6,dim7,dim8,dim9,dim10,...,dim247,dim248,dim249,dim250,dim251,dim252,dim253,dim254,dim255,dim256
ID,,,,,,,,,,,,,,,,,,,,,
sub-1000021,144.70032,-2.33259,-55.521217,-3.811033,-44.061070,41.837160,15.210489,-3.336162,-16.129618,-20.501787,...,-12.813190,-1.744127,87.089485,15.318643,-43.634010,40.511910,21.935389,5.046747,-9.521525,-53.306515
sub-1000325,189.53354,31.76580,18.947166,-8.772030,20.106237,-33.390730,17.316332,-29.190847,20.369950,-5.060660,...,17.064726,-24.591824,33.550570,47.255363,-66.175830,47.804210,-30.778520,9.909952,-13.292032,-40.064840
sub-1000458,150.20581,-1.78613,-26.557894,8.816937,-41.133255,136.199750,-2.472998,12.573650,12.101191,22.242050,...,-11.313081,5.824224,8.352239,13.173943,23.730171,-16.312475,-6.631551,6.091625,20.440187,-85.629180
sub-1000575,145.42535,-49.58505,8.499626,12.548178,-20.224136,76.969540,4.202240,-19.305752,-32.917637,25.650387,...,-14.098648,-0.068690,-18.289234,-10.241393,18.584848,-12.241503,33.521553,1.713032,8.407868,-88.648970
sub-1000606,153.36888,29.05402,-11.328059,56.158030,-35.677720,13.826549,-7.284871,21.386320,29.033829,-8.862112,...,-15.195081,14.316205,36.025223,43.307170,1.105324,-9.904149,-24.226252,-0.112310,-18.451952,2.476232


### Reduce dimension (hope to remove the noise) with a PCA

In [162]:
n_components=40

pca = PCA(n_components=n_components)
pca.fit(ukb_embeddings)
print(pca.explained_variance_ratio_)
(np.cumsum(pca.explained_variance_ratio_) < 0.99).sum()

[1.75927055e-01 1.23061051e-01 1.17469308e-01 1.10334597e-01
 1.07558876e-01 9.08443633e-02 7.94235493e-02 7.29577511e-02
 4.26399328e-02 2.65265101e-02 1.56362286e-02 1.14035315e-02
 6.37187943e-03 5.82675762e-03 3.07860820e-03 2.18080580e-03
 1.81282230e-03 1.44148820e-03 1.05229146e-03 8.57490510e-04
 5.91649111e-04 5.23398021e-04 3.99521802e-04 3.66117464e-04
 2.07805719e-04 2.02041088e-04 1.51386990e-04 1.27990357e-04
 1.06812169e-04 1.00005020e-04 9.43594938e-05 7.45294226e-05
 6.16218394e-05 4.99351672e-05 4.31299419e-05 4.08603383e-05
 3.44245619e-05 3.15927279e-05 2.90202554e-05 2.53794225e-05]


15

In [163]:
ukb_pca_bdd = pca.transform(ukb_embeddings)

In [164]:
#scaler = StandardScaler()
#scaler.fit(ukb_embeddings)
#ukb_scl_bdd = scaler.transform(ukb_embeddings)
#ukb_scl_bdd

#### First approach: SVM trained to find the interruption

In [165]:
model = SVC(kernel='linear', probability=True,
            random_state=42,
            C=0.01, class_weight='balanced', decision_function_shape='ovr')

In [179]:
interrupted = [
'sub-1310920',
'sub-1376904',
'sub-2863742',
'sub-3694216',
'sub-1037052',
'sub-3250551',
'sub-5401486',
'sub-1499791', # good
'sub-1911266',
'sub-4217758',
'sub-2693192',
'sub-1633860',
'sub-5222070',
'sub-3292254',
'sub-1613821',
'sub-2771619',
'sub-3159828',
'sub-4632483',
'sub-5936108',
'sub-3794487',
'sub-1420697', # not sure
'sub-1111996', # not sure
'sub-1425827', # not sure
'sub-2846621', # good
'sub-2004479',
'sub-3891499',
'sub-5236788',
'sub-3061407', # very good
'sub-5693167',
'sub-2155264', # very good
'sub-2444973', # very good
'sub-5245412', # good
'sub-5574911', # very good
'sub-2852894', # very good
'sub-1106033', # very good
'sub-5984646', # very good
'sub-5739487', # very good
'sub-3492298', # good
'sub-5712569', # not sure
'sub-2200121', # not sure
'sub-5638090', # good
'sub-4496792', # good
'sub-5129881', # good
'sub-1775041', # good
'sub-1094593', # good
'sub-1358401', # good
'sub-4354208', # very good
'sub-1428452', # good
'sub-5731125',
'sub-4995189', # very good
'sub-1499791',
'sub-2762943', # very good
'sub-3386408', # not sure
'sub-5665554', # not sure
'sub-1130686', # good
'sub-2484762', # good
'sub-5186095', # good
'sub-5569356',
'sub-4762603', # not sure
'sub-3572724', # good
'sub-2573795', # good
'sub-5315648',
'sub-2731992',
'sub-4949978',
'sub-2776534',
'sub-2298245',
'sub-2570335',
'sub-3258249', # not sure
'sub-1748817', # not sure
'sub-4203366', # very good
'sub-4184635', # very good
'sub-1053493', # note sure
'sub-3947538', # not sure
'sub-2741631', # not sure
'sub-3492301',
'sub-4447456',
'sub-2373286', # not sure
'sub-4732282',
'sub-3293670',
'sub-2149638', # good
'sub-4625643', # not sure
'sub-4328267',
'sub-2589361',
'sub-4232003',
'sub-5456948',
'sub-4420000', # not sure
'sub-1553423', # not sure
'sub-1405899', # not sure
'sub-2550690', # not sure
'sub-2986522', # not sure
'sub-1698233', # not sure
'sub-4603077', # not sure
'sub-3428215',
'sub-1935008',
'sub-4589882',
'sub-2323818',
'sub-2230154',
'sub-1675253',
'sub-4875056',
'sub-4059279',
'sub-4067363',
'sub-1322441',
'sub-1417407',
'sub-3733675',
'sub-5531350',
'sub-4067363',
'sub-1369171',
'sub-1807186',
'sub-1267836',
'sub-3758439',
'sub-4652131',
'sub-1052521',
'sub-5949398',
'sub-3672666',
'sub-4754998',
'sub-3791185',
'sub-4587270',
'sub-4599903',
'sub-5617588',
'sub-1428212',
'sub-3911620',
'sub-4152006',
'sub-1864685',
'sub-5366951',
'sub-3679537',
'sub-5209589',
'sub-4211996',
'sub-3913796',
'sub-5777436',
'sub-4340260',
'sub-1132414',
'sub-5428293',
'sub-5406975',
'sub-4286421',
'sub-3253763',
'sub-1154509',
'sub-3500106',
'sub-4779638',
]

not_interrupted = [
'sub-1103646',
 'sub-1167379',
 'sub-1190643',
 'sub-1273718',
 'sub-1286007',
 'sub-1298876',
 'sub-1352284',
 'sub-1398736',
 'sub-1422413',
 'sub-1465129',
 'sub-1597706',
 'sub-1701563',
 'sub-1734788',
 'sub-1979982',
 'sub-1996092',
 'sub-2005939',
 'sub-2036033',
 'sub-2097565',
 'sub-2118136',
 'sub-2141551',
 'sub-2193253',
 'sub-2207793',
 'sub-2228486',
 'sub-2284024',
 'sub-2337820',
 'sub-2349203',
 'sub-2389411',
 'sub-2420937',
 'sub-2427515',
 'sub-2538754',
 'sub-2583027',
 'sub-2592717',
 'sub-2733674',
 'sub-2741815',
 'sub-2792782',
 'sub-2802489',
 'sub-2814161',
 'sub-2816262',
 'sub-2833426',
 'sub-2834970',
 'sub-2837393',
 'sub-2889389',
 'sub-2946274',
 'sub-2957401',
 'sub-2968297',
 'sub-2970418',
 'sub-3008660',
 'sub-3009279',
 'sub-3013938',
 'sub-3227039',
 'sub-3234836',
 'sub-3264612',
 'sub-3333294',
 'sub-3334219',
 'sub-3379262',
 'sub-3388080',
 'sub-3388306',
 'sub-3401499',
 'sub-3453064',
 'sub-3525594',
 'sub-3529189',
 'sub-3541105',
 'sub-3603191',
 'sub-3627711',
 'sub-3670173',
 'sub-3693543',
 'sub-3721299',
 'sub-3722413',
 'sub-3765466',
 'sub-3936967',
 'sub-3992259',
 'sub-3994474',
 'sub-4016129',
 'sub-4027732',
 'sub-4116944',
 'sub-4411765',
 'sub-4420611',
 'sub-4428393',
 'sub-4491384',
 'sub-4519441',
 'sub-4520944',
 'sub-4536778',
 'sub-4727825',
 'sub-4741296',
 'sub-4755899',
 'sub-4787289',
 'sub-4791977',
 'sub-4805119',
 'sub-4805237',
 'sub-4834994',
 'sub-4868991',
 'sub-5027399',
 'sub-5054716',
 'sub-5082433',
 'sub-5117110',
 'sub-5123219',
 'sub-5147403',
 'sub-5217534',
 'sub-5237880',
 'sub-5292898',
 'sub-5293703',
 'sub-5319071',
 'sub-5430535',
 'sub-5437419',
 'sub-5486726',
 'sub-5561142',
 'sub-5578922',
 'sub-5581707',
 'sub-5605784',
 'sub-5643778',
 'sub-5649675',
 'sub-5686761',
 'sub-5723111',
 'sub-5729132',
 'sub-5749108',
 'sub-5754849',
 'sub-5836983',
 'sub-5864979',
 'sub-5910947',
 'sub-5966409',
 'sub-5998652',
 'sub-5357627',
 'sub-2204575',
 'sub-2839753',
 'sub-4281714',
 'sub-1649070',
 'sub-5335727',
 'sub-5782466',
 'sub-4520082',
 'sub-1004170',
 'sub-4158073', 
 'sub-5684893',
 'sub-4359496',
 'sub-2040983',
 'sub-5575777',
 'sub-1116938',
 'sub-4189639',
 'sub-4507392',
 'sub-5085553',
 'sub-2077194',
 'sub-5457081',
 'sub-3692612',
 'sub-2379487',
 'sub-5137278',
 'sub-4831688',
 'sub-3976041',
 'sub-4057189',
 'sub-4202490',
 'sub-4844615',
 'sub-4747425',
 'sub-1008582',
 'sub-4039492',
] 

In [180]:
X = ukb_embeddings.loc[interrupted + not_interrupted]
y = [1 for i in range(len(interrupted))] + [0 for i in range(len(not_interrupted))]
X_pca = pca.transform(X)
len(interrupted), len(not_interrupted)

(133, 149)

In [181]:
X_train_pca, X_test_pca, y_train, y_test = train_test_split(X_pca, y, test_size=0.33, random_state=42)
model.fit(X_train_pca, y_train)

print('Recall:', recall_score(y_test, model.predict(X_test_pca)), '\n')
print('ROC:', roc_auc_score(y_test ,model.predict_proba(X_test_pca)[:,1]), '\n')
print('Balanced accuracy:',balanced_accuracy_score(y_test, model.predict(X_test_pca)), '\n')
model.fit(X_pca, y)

Recall: 0.8085106382978723 

ROC: 0.7682209144409236 

Balanced accuracy: 0.6808510638297872 



SVC(C=0.01, class_weight='balanced', kernel='linear', probability=True,
    random_state=42)

In [182]:
prediction = pd.DataFrame({"IID" : list(ukb_embeddings.index),
              "Pred" : model.predict_proba(ukb_pca_bdd)[:,1]})
prediction

,IID,Pred
0,sub-1000021,0.334685
1,sub-1000325,0.085015
2,sub-1000458,0.278749
3,sub-1000575,0.282997
4,sub-1000606,0.136538
...,...,...
42428,sub-6023847,0.226215
42429,sub-6024038,0.569368
42430,sub-6024150,0.196867
42431,sub-6024379,0.300565


In [170]:
print('Maximum probability of prediction among the interrupted C.S. :',prediction[prediction["IID"].isin(interrupted)].Pred.max(), '\n')
print('Mean probability of prediction among the interrupted C.S. :',prediction[prediction["IID"].isin(interrupted)].Pred.mean(), '\n')
prediction[prediction['IID']=='sub-2036033']

Maximum probability of prediction among the interrupted C.S. : 0.9438194028972571 

Mean probability of prediction among the interrupted C.S. : 0.6179740508753789 



,IID,Pred
8695,sub-2036033,0.476616


In [49]:
((prediction[~(prediction["IID"].isin(interrupted))]).sort_values(by="Pred")[-5:].IID).to_list()

['sub-2223729', 'sub-2461363', 'sub-3566456', 'sub-1366715', 'sub-5723524']

#### Second approach: Euclidian distance in the reduced latent space

In [13]:
from scipy.spatial import distance

In [1097]:
list_dist = [distance.euclidean(pca.transform(ukb_embeddings.loc['sub-3791185'].to_numpy().reshape(1,-1)), ukb_pca_bdd[i]) for i in range(len(ukb_pca_bdd))]
df_dist = pd.DataFrame({"IID":list(ukb_embeddings.index), "Dist":list_dist})

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr

In [1098]:
sample_dist = ((df_dist[~(df_dist["IID"].isin(interrupted))]).sort_values(by='Dist').iloc[22000:22025].IID).to_list()

### Visualization with Anatomist

In [14]:
import anatomist.api as ana
from soma.qt_gui.qtThread import QtThreadCall
from soma.qt_gui.qt_backend import Qt

a = ana.Anatomist()

from soma import aims

create qapp
global modules: /casa/host/build/share/anatomist-5.2/python_plugins
home   modules: /casa/home/.anatomist/python_plugins
loading module simple_controls
done
Starting Anatomist.....
config file : /casa/home/.anatomist/config/settings.cfg
PyAnatomist Module present
PythonLauncher::runModules()
loading module save_resampled


existing QApplication: 0
QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-ad279118'


loading module selection
loading module bsa_proba
loading module modelGraphs
loading module profilewindow
loading module ana_image_math
loading module paletteViewer
loading module foldsplit
loading module anacontrolmenu
loading module gradientpalette
loading module palettecontrols
loading module meshsplit
loading module volumepalettes
loading module gltf_io
loading module infowindow
loading module histogram
loading module statsplotwindow
loading module valuesplotwindow
all python modules loaded
Anatomist started.


In [50]:
dataset = 'UkBioBank40'
region = "S.C.-sylv."
side = "R"

mm_skeleton_path = f'/neurospin/dico/data/deep_folding/current/datasets/{dataset}/crops/2mm/{region}/mask/{side}crops'

In [187]:
sample = ((prediction[~(prediction["IID"].isin(interrupted))]).sort_values(by="Pred", ascending=False)[50:75].IID).to_list()

In [188]:
volume=True
volume_files = []

for subject_id in sample:
    volume_path = f"{mm_skeleton_path}/{subject_id}_cropped_skeleton.nii.gz"
    
    if volume:
        if os.path.isfile(volume_path):
            vol = aims.read(volume_path)
            volume_files.append(vol)
        else:
            print(f"{volume_path} is not a correct path, or the .nii.gz doesn't exist")

block = a.createWindowsBlock(5) # 10 columns
dic_windows = {}

if volume:
    for i, vol in enumerate(volume_files):
        dic_windows[f'a_vol{i}'] = a.toAObject(vol)
        #dic_windows[f'a_vol{i}'].setPalette(absoluteMode=True)
        dic_windows[f'rvol{i}'] = a.fusionObjects(objects=[dic_windows[f'a_vol{i}']], method='VolumeRenderingFusionMethod')
        dic_windows[f'rvol{i}'].releaseAppRef()
        dic_windows[f'wvr{i}'] = a.createWindow('3D', block=block) #geometry=[100+400*(i%3), 100+440*(i//3), 400, 400])
        dic_windows[f'wvr{i}'].addObjects(dic_windows[f'rvol{i}'])

no position could be read at 282, 109
no position could be read at 251, 117
no position could be read at 226, 120
no position could be read at 167, 101
no position could be read at 248, 99
no position could be read at 215, 100
no position could be read at 209, 87
no position could be read at 212, 102


In [194]:
sample[15]

'sub-4039492'

In [1077]:
sample_dist[15:19]

['sub-5293703', 'sub-5319071', 'sub-3525594', 'sub-5561142']